In [0]:
df=spark.read.csv("/Volumes/workspace/default/raw_data/orders.csv")
display(df)

from pyspark.sql import types as T, functions as F
csv_schema = T.StructType([
    T.StructField("order_date", T.DateType(), True), 
    T.StructField("country", T.StringType(), True),
    T.StructField("order_id", T.IntegerType(), True),
    T.StructField("product", T.StringType(), True),
    T.StructField("qty", T.IntegerType(), True),
    T.StructField("price", T.DoubleType(), True),
])

In [0]:
df = spark.read.option("header", "true") \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(csv_schema) \
    .csv("/Volumes/workspace/default/raw_data/orders.csv")

display(df)

In [0]:
#overwriting to a file
df.write.mode("overwrite").parquet("/Volumes/workspace/default/raw_data/orders")


### SQL USING Python

In [0]:
df_marvel=spark.sql("select * from workspace.default.movies where studio='Marvel Studios'")
df_marvel.show()

In [0]:
%sql
select * from workspace.default.movies where studio='Marvel Studios'


In [0]:
#creating from dataframe
from pyspark.sql import functions as F, types as T

data=[
    ("2017-01-01",32.0,6.0,"Rain"),
    ("2017-01-04",None,9.0,"Sunny"),
    ("2017-01-05",28.0,None,"Snow"),
    ("2017-01-06",None,7.0,None),
    ("2017-01-07",32.0,None,"Rain"),
    ("2017-01-08",None,12.0,"Sunny"),
    ("2017-01-09",None,None,None),
    ("2017-01-10",34.1,8.1,"Cloudy"),
    ("2017-01-11",40.0,12.0,"Sunny"),
]
schema="day string, temperature double, windspeed double, event string"
df=spark.createDataFrame(data,schema)
df=df.withColumn("day",F.to_date("day","yyyy-MM-dd"))  #normalizing to datatype
display(df)


In [0]:
df.createOrReplaceTempView("weather")   # only for current pyspark session
df.createGlobalTempView("global_weather")    #another notebook even in same cluster can work on it

In [0]:
%sql
select event, ROUND(avg(temperature), 1) as avg_temp
from weather
group by event
order by avg_temp desc;

##Performing Joins in spark

In [0]:
from pyspark.sql import functions as F, types as T

rows_customers=[
    (1,"Asha","IN",True),
    (2,"Bob","US",False),
    (3,"Chen","CN",True),
    (4,"Diana","US",None),
    (None,"Ghost","UK",False),  
]

rows_orders = [
    (101,1,120.0,"IN"),
    (102,1,80.0,"IN"),
    (103,2,50.0,"US"),
    (104,5,30.0,"DE"),
    (105,3,200.0,"CN"),
    (106,None,15.0,"UK"),
    (107,3,40.0,"CN"),
    (108,2,75.0,"US"),
]


schema_customers = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True), 
    T.StructField("name", T.StringType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("vip", T.BooleanType(), True),
])
schema_orders = T.StructType([
    T.StructField("order_id", T.IntegerType(), True), 
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("country", T.StringType(), True),
])
df_customers = spark.createDataFrame(rows_customers,schema_customers)
df_orders = spark.createDataFrame(rows_orders,schema_orders)
display(df_customers)
display(df_orders)

In [0]:
df_inner = df_orders.join(df_customers,on="customer_id",how="inner")
display(df_inner)

In [0]:
df_inner = df_orders.join(df_customers,on="customer_id",how="left")
display(df_inner)

In [0]:
o=df_inner.alias("o")
o.show()

In [0]:
o,c=df_orders.alias("o"),df_customers.alias("c")
o.show()
c.show()
df_inner= o.join(c,on="customer_id",how="inner")
display(df_inner)





In [0]:
df_inner_clean=(
    o.join(c,on="customer_id",how="inner")
    .select("order_id","customer_id","amount",
    F.col("o.country").alias("ship_country")
    ,"name",
     F.col("c.country").alias("cust_country"),
     "vip")
)
display(df_inner_clean)

In [0]:
df=spark.table("workspace.default.movies")
df.show(3)

In [0]:
from pyspark.sql import functions as F, types as T
df_narrow=df.select("title","studio","imdb_rating").filter(F.col("release_year")>2010)
display(df_narrow)

In [0]:
df_narrow.explain("extended")

In [0]:
df_narrow.explain("formatted")

In [0]:
#Transformation happens here
df=spark.read.csv("data.csv")
df_filtered=df.filter(col("age")>18)
df_selected=df_filtered.select("name")

#action-----now everything executes
df_selected.show()  #action triggers all above executions